# Track 3 v2, stage 1: abstention SFT + a 90-task holdout

Three of the four weaknesses in the submission are GPU-shaped, and this kernel fixes two:

1. **Abstention is open at 1/8.** The frozen adapter invents participants when the user never
   gave them. `data/build_abstain.py` adds 32 underspecified rows as minimal pairs (the same
   expense with the missing field withheld, plus a twin where it is present) and
   `data/build_mix.py --abstain` folds them into a 312-row SFT mix. Asking is 8.4% of the
   supervised tokens and acting stays 81.3%, so this is a targeted patch, not a policy flip.
2. **hard-30 is 30 tasks.** `data/build_hard90.py` builds a fresh 90-task split, 15 per family,
   pinned by its own SHA-256, deduplicated against hard-30, hard-train, easy, safety and the
   abstention split. Same frozen adapter, three times the grading signal.

**Wall clock on T4 x2 (~7.8 h of the 12 h budget).** Cells are ordered so the cheap, critical
numbers land first: if the session dies late, you still have hard-30, eval-24, safety and the
seed-2 checkpoint.

| step | time |
|---|---:|
| build + verify | 4 min |
| train `sft3` on mix2, 2 epochs, seed 42 | 65 min |
| `sft3` on hard-30, eval-24, safety | 75 min |
| `sft3` on hard-90 | 90 min |
| `sft2` (frozen) on hard-90 | 90 min |
| base on hard-90 | 120 min |

Uploads: the `sft3` adapter to `primce/tiny-aya-hinglish-lora`, every run directory to
`primce/lexsi-runs`. Paste the printed summaries back into chat; the docs are filled from
`summary.json`, never from estimates.

In [ ]:
import os
from pathlib import Path

from kaggle_secrets import UserSecretsClient

try:
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    raise SystemExit(
        "HF_TOKEN is not attached. CohereLabs/tiny-aya-global is gated, so the token is "
        "needed to load the model, not only to upload. Add-ons -> Secrets, then Save & Run All."
    ) from exc

os.environ["HF_TOKEN"] = token
# bash cells cannot see this Python env; hand them the token through a private tmp file
# instead of the kernel output log.
env = Path("/tmp/hf_env")
env.write_text(f'export HF_TOKEN="{token}"\n', encoding="utf-8")
env.chmod(0o600)
print("HF_TOKEN set (not printed)")

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
set -euo pipefail
source /tmp/hf_env
cd /kaggle/working
rm -rf lexsi-agentic-hinglish
git clone -q https://github.com/PRINCE2-AI/lexsi-agentic-hinglish.git
cd lexsi-agentic-hinglish
git log --oneline -1
python -m pip install -q -r requirements.txt
python scripts/verify_all.py

## Stage 1: train `sft3`

Same config as the frozen `sft2` (LoRA r=16 all-linear, fp16, lr 2e-4, 2 epochs) so the only
change is the data. `--seed 42` matches the frozen run, which makes the later seed-123 run a
clean variance measurement rather than a confound.

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
cd /kaggle/working/lexsi-agentic-hinglish
time python scripts/train_sft.py --tasks data/tasks_train_mix2.jsonl --epochs 2 --seed 42 \
  --out runs/sft3 --merge-out runs/sft3_merged || echo "SFT3 TRAIN FAILED"

## Stage 2: evaluate `sft3`

hard-30 is unchanged and still frozen, so the headline before/after stays comparable. The
abstention rows are 8.4% of supervised tokens, so the first thing to check is that hard-30 did
not drop.

In [ ]:
%%bash
cd /kaggle/working/lexsi-agentic-hinglish
set -uo pipefail
source /tmp/hf_env
if [ ! -f runs/sft3_merged/config.json ]; then echo "SFT3 MERGE MISSING"; exit 0; fi

CUDA_VISIBLE_DEVICES=0 python scripts/run_baseline.py --model runs/sft3_merged \
  --tasks data/tasks_hard_eval.jsonl --no-4bit --out runs/sft3_hard30 || echo "SFT3 HARD30 FAILED"

CUDA_VISIBLE_DEVICES=0 python scripts/run_baseline.py --model runs/sft3_merged \
  --tasks data/tasks_eval.jsonl --no-4bit --out runs/sft3_eval || echo "SFT3 EVAL24 FAILED"

CUDA_VISIBLE_DEVICES=0 python scripts/run_baseline.py --model runs/sft3_merged \
  --tasks data/tasks_safety_eval.jsonl --safety --no-4bit --out runs/safety_sft3 || echo "SFT3 SAFETY FAILED"

CUDA_VISIBLE_DEVICES=0 python scripts/run_baseline.py --model runs/sft3_merged \
  --tasks data/tasks_safety_eval.jsonl --safety --hardened-prompt --no-4bit \
  --out runs/safety_sft3_hardened || echo "SFT3 HARDENED FAILED"

In [ ]:
%%bash
cd /kaggle/working/lexsi-agentic-hinglish
set -uo pipefail
source /tmp/hf_env
if [ ! -f runs/sft3_merged/config.json ]; then echo "SFT3 MERGE MISSING"; exit 0; fi
CUDA_VISIBLE_DEVICES=0 python scripts/run_baseline.py --model runs/sft3_merged \
  --tasks data/tasks_hard90_eval.jsonl --no-4bit --out runs/sft3_hard90 || echo "SFT3 HARD90 FAILED"

## Stage 3: the frozen `sft2` on hard-90

No retraining: merge the adapter that is already on the Hub, then score it on the new split. This
is the row the `sft3` number has to beat, on 90 tasks instead of 30.

In [ ]:
import torch
from huggingface_hub import snapshot_download
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = "CohereLabs/tiny-aya-global"
repo = snapshot_download("primce/tiny-aya-hinglish-lora", allow_patterns=["sft2_adapter/*"],
                         local_dir="/kaggle/working/sft2_repo")
MERGED = "/kaggle/working/sft2_merged"
print("adapter repo:", repo)

tokenizer = AutoTokenizer.from_pretrained(BASE)
for path, subfolder in ((MERGED, "sft2_adapter"),):
    model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="auto")
    model = PeftModel.from_pretrained(model, repo, subfolder=subfolder)
    merged = model.merge_and_unload()
    merged.save_pretrained(path, safe_serialization=True)
    tokenizer.save_pretrained(path)
    del model, merged
    torch.cuda.empty_cache()
    print("merged ->", path)

In [ ]:
%%bash
cd /kaggle/working/lexsi-agentic-hinglish
set -uo pipefail
source /tmp/hf_env
if [ ! -f /kaggle/working/sft2_merged/config.json ]; then echo "SFT2 MERGE MISSING"; exit 0; fi
CUDA_VISIBLE_DEVICES=0 python scripts/run_baseline.py --model /kaggle/working/sft2_merged \
  --tasks data/tasks_hard90_eval.jsonl --no-4bit --out runs/sft2_hard90 || echo "SFT2 HARD90 FAILED"

## Stage 4 (optional, last): base Tiny Aya on hard-90

Only run this if the session still has time. It gives the new split its own prompt-only row, so
the hard-90 table reads base -> sft2 -> sft3 like the hard-30 one. Base averages 81 s/task, so
expect ~2 h.

In [ ]:
%%bash
cd /kaggle/working/lexsi-agentic-hinglish
set -uo pipefail
source /tmp/hf_env
CUDA_VISIBLE_DEVICES=0 python scripts/run_baseline.py --model CohereLabs/tiny-aya-global \
  --tasks data/tasks_hard90_eval.jsonl --no-4bit --out runs/base_hard90 || echo "BASE HARD90 FAILED"

## Collect and upload

In [ ]:
import json
import os
from pathlib import Path

from huggingface_hub import HfApi

ROOT = Path("/kaggle/working/lexsi-agentic-hinglish")
os.chdir(ROOT)

RUNS_REPO = "primce/lexsi-runs"
MODEL_REPO = "primce/tiny-aya-hinglish-lora"
RUNS = ['sft3_hard30', 'sft3_eval', 'sft3_hard90', 'safety_sft3', 'safety_sft3_hardened', 'sft2_hard90', 'base_hard90']
KEYS = (
    "n", "success", "tac", "arr", "ter", "rad", "overcall", "lcf", "answer_ok",
    "strict_rate", "calls", "steps", "parse_failures", "seconds",
    "safe", "utility", "attack_ledger", "attack_answer", "unplanned_calls",
    "abstained", "asked", "invented_expense", "injections_fired",
)

for name in RUNS:
    path = ROOT / "runs" / name / "summary.json"
    if not path.exists():
        print(f"{name}: MISSING")
        continue
    summary = json.loads(path.read_text())
    print(name, {key: summary[key] for key in KEYS if key in summary})
    families = summary.get("families") or {}
    for family, row in families.items():
        print(f"   {family}: success={row.get('success')} safe={row.get('safe')}")

token = os.environ.get("HF_TOKEN")
if not token:
    print("no HF_TOKEN in this kernel, skipping upload")
else:
    api = HfApi(token=token)
    for name in RUNS:
        folder = ROOT / "runs" / name
        if folder.exists():
            api.upload_folder(repo_id=RUNS_REPO, repo_type="dataset",
                              folder_path=str(folder), path_in_repo=name)
            print(f"{name} -> {RUNS_REPO}")
    adapter = ROOT / "runs" / "sft3"
    if (adapter / "adapter_config.json").exists():
        api.upload_folder(repo_id=MODEL_REPO, folder_path=str(adapter), path_in_repo="sft3_adapter",
                          commit_message="sft3: mix2 + abstention split, seed 42")
        print("sft3_adapter ->", MODEL_REPO)

Paste the printed block back into chat. What the docs need from it:

- `sft3_hard30.success` — did the abstention data cost anything on the frozen split?
- `safety_sft3.asked` / `abstained` — the number that is 1/8 today.
- `sft3_hard90.success` vs `sft2_hard90.success` — the 90-task comparison.
- `seconds` for every run, to keep the time table honest.